# "But wait... there's more"

## A More Visible Agent Loop

The Digital Twin contained an Agent Loop. But it was behind-the-scenes, running every time the user asked a message. Using its tools and then replying. It didn't feel very... loopy.

### Adding 2 more ingredients to make it more real

Let's make an Agent Loop with some familiar features borrowed from Claude Code:

1. A Terminal UI (TUI)
2. A Checklist tool to cause and track multiple tool calls


In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import os
import json
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter_base_url= os.getenv('OPENROUTER_BASE_URL')
openai = OpenAI(base_url=openrouter_base_url,api_key=openrouter_api_key )

In [4]:
# Some lists!

checklist = []
completed = []

In [5]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [6]:
get_checklist_report()

''

In [7]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [8]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [9]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: Buy groceries\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [10]:
mark_complete(1, "bought")

bought

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [11]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [12]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [13]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [14]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [15]:
def loop(messages):
    response = openai.chat.completions.create(model="openrouter/free", messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="openrouter/free", messages=messages, tools=tools)
    show(response.choices[0].message.content)

In [16]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [17]:
checklist, completed = [], []
loop(messages)

Checklist #1: Estimate the distance between Boston and New York
Checklist #2: Compute the relative speed of the two trains
Checklist #3: Determine the time elapsed from 3:00 pm until the trains meet
Checklist #4: Compute the meeting time

Estimated the distance between Boston and New York as approximately 215 miles (typical driving distance).

Checklist #1: Estimate the distance between Boston and New York
Checklist #2: Compute the relative speed of the two trains
Checklist #3: Determine the time elapsed from 3:00 pm until the trains meet
Checklist #4: Compute the meeting time

Computed relative speed: 60 mph + 80 mph = 140 mph.

Checklist #1: Estimate the distance between Boston and New York
Checklist #2: Compute the relative speed of the two trains
Checklist #3: Determine the time elapsed from 3:00 pm until the trains meet
Checklist #4: Compute the meeting time

Determined that the first train left at 2:00 pm, so by 3:00 pm it had traveled 60 miles. Remaining distance at 3:00
pm = 215 - 60 = 155 miles. Time to meet after 3:00 pm = 155 / 140 ≈ 1.107 hours (≈ 1 hour 6.4 minutes).

Checklist #1: Estimate the distance between Boston and New York
Checklist #2: Compute the relative speed of the two trains
Checklist #3: Determine the time elapsed from 3:00 pm until the trains meet
Checklist #4: Compute the meeting time

Meeting time: 3:00 pm + 1 hour 6.4 minutes ≈ 4:06 pm.

Checklist #1: Estimate the distance between Boston and New York
Checklist #2: Compute the relative speed of the two trains
Checklist #3: Determine the time elapsed from 3:00 pm until the trains meet
Checklist #4: Compute the meeting time

**Solution:**

1. **Estimate the distance between Boston and New York:**  
   The typical driving distance is approximately 215 miles. (This is a reasonable estimate; the actual rail 
distance may vary slightly but is not provided in the problem.)

2. **Compute the relative speed:**  
   The trains are moving toward each other, so their speeds add:  
   \(60 \text{ mph} + 80 \text{ mph} = 140 \text{ mph}\).

3. **Determine the time elapsed from 3:00 pm until they meet:**  
   - The Boston train leaves at 2:00 pm, so by 3:00 pm it has traveled \(60 \text{ mph} \times 1 \text{ hour} = 60 
\text{ miles}\).  
   - Remaining distance at 3:00 pm: \(215 - 60 = 155 \text{ miles}\).  
   - Time to cover 155 miles at 140 mph:  
     [
     t = \frac{155}{140} \approx 1.107 \text{ hours} \approx 1 \text{ hour } 6.4 \text{ minutes}.
     \]

4. **Compute the meeting time:**  
   Starting from 3:00 pm, add the time calculated:  
   \(3:00 \text{ pm} + 1 \text{ hour } 6.4 \text{ minutes} \approx 4:06 \text{ pm}\).

**Answer:**  
The trains meet at approximately **4:06 pm**.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>

In [20]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import os
import json
load_dotenv(override=True)


checklist = []
completed = []

model = "openrouter/free"
base_url = os.getenv('OPENROUTER_BASE_URL')
api_key  = os.getenv('OPENROUTER_API_KEY')

openai = OpenAI(base_url = base_url, api_key= api_key)

create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}


mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}


def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result


def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()



tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]


def handle_tool_calls(tool_calls): 
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

def loop(messages):
    response = openai.chat.completions.create(model=model, messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="openrouter/free", messages=messages, tools=tools)
    show(response.choices[0].message.content)



system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]



loop(messages=messages)


Checklist #1: Determine the distance between Boston and New York (assume 225 miles)
Checklist #2: Calculate distance traveled by first train from 2:00 PM to 3:00 PM
Checklist #3: Find remaining distance between trains at 3:00 PM
Checklist #4: Calculate combined speed of both trains
Checklist #5: Compute time until they meet after 3:00 PM
Checklist #6: Add meeting time to 3:00 PM to get final meeting time

Assume Boston to New York distance is 225 miles

Checklist #1: Determine the distance between Boston and New York (assume 225 miles)
Checklist #2: Calculate distance traveled by first train from 2:00 PM to 3:00 PM
Checklist #3: Find remaining distance between trains at 3:00 PM
Checklist #4: Calculate combined speed of both trains
Checklist #5: Compute time until they meet after 3:00 PM
Checklist #6: Add meeting time to 3:00 PM to get final meeting time

First train leaves Boston at 2:00 PM at 60 mph. In 1 hour until 3:00 PM, it travels 60 × 1 = 60 miles toward New 
York.

Checklist #1: Determine the distance between Boston and New York (assume 225 miles)
Checklist #2: Calculate distance traveled by first train from 2:00 PM to 3:00 PM
Checklist #3: Find remaining distance between trains at 3:00 PM
Checklist #4: Calculate combined speed of both trains
Checklist #5: Compute time until they meet after 3:00 PM
Checklist #6: Add meeting time to 3:00 PM to get final meeting time

...

Remaining distance 165 miles divided by combined speed 140 mph = 165/140 hours = 1.1786 hours ≈ 1 hour 10.7 minutes
≈ 1 hour 11 minutes

Checklist #1: Determine the distance between Boston and New York (assume 225 miles)
Checklist #2: Calculate distance traveled by first train from 2:00 PM to 3:00 PM
Checklist #3: Find remaining distance between trains at 3:00 PM
Checklist #4: Calculate combined speed of both trains
Checklist #5: Compute time until they meet after 3:00 PM
Checklist #6: Add meeting time to 3:00 PM to get final meeting time

3:00 PM + 1 hour 11 minutes = 4:11 PM (approximately). They meet at approximately 4:11 PM.

Checklist #1: Determine the distance between Boston and New York (assume 225 miles)
Checklist #2: Calculate distance traveled by first train from 2:00 PM to 3:00 PM
Checklist #3: Find remaining distance between trains at 3:00 PM
Checklist #4: Calculate combined speed of both trains
Checklist #5: Compute time until they meet after 3:00 PM
Checklist #6: Add meeting time to 3:00 PM to get final meeting time

**Solution:**

**Assumption:** Distance between Boston and New York = 225 miles

**Working:**

1. **2:00 PM to 3:00 PM:** Train from Boston travels 60 mph × 1 hour = **60 miles**

2. **At 3:00 PM:** Remaining distance between trains = 225 - 60 = **165 miles**

3. **After 3:00 PM:** Both trains approaching each other, so combined speed = 60 + 80 = **140 mph**

4. **Time to meet after 3:00 PM:** 165 miles ÷ 140 mph = **1.1786 hours ≈ 1 hour 11 minutes**

**Answer:** The trains meet at approximately **4:11 PM**

---

*Note: If the Boston-New York distance is assumed to be different (commonly cited values range from 150-230 miles),
the meeting time would adjust proportionally.*